# 79. The Freight Carrier Selection & Bidding Problem
## Tier 3: The Advanced Algorithm (Metaheuristic Implementation)

### Key assumptions
- Simulated Annealing allows temporary acceptance of worse solutions
- Temperature-controlled search process balances exploration and exploitation
- Neighborhood structure explores adjacent carrier assignments
- Convergence to near-optimal solutions without guaranteeing optimality

### Approach (step-by-step)
1. **Initialize solution** with random or greedy carrier assignments
2. **Set temperature schedule** with initial temperature and cooling rate
3. **Generate neighbor solutions** by modifying current assignments
4. **Calculate acceptance probability** based on solution improvement and temperature
5. **Iteratively improve solution** while gradually reducing temperature
6. **Track convergence** and monitor solution quality over iterations

### What to look for in the results
- Convergence behavior showing improvement over iterations
- Temperature decay allowing focused search in later stages
- Better solutions than greedy heuristic with reasonable computation time
- Trade-off between solution quality and computational effort

### Concrete example (from the source)
Simulated Annealing with 5000 iterations:
- Initial temperature: 1000, cooling rate: 0.995
- Expected optimal assignment: Lane 1 → Carrier 2, Lane 2 → Carrier 3
- Expected cost improvement: 2.3% over greedy solution
- Expected convergence at iteration ~3,247

In [1]:
# Import required libraries for simulated annealing
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time as time_module
import random
import warnings
warnings.filterwarnings('ignore')

# Set up visualization style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Set random seed for reproducible results
np.random.seed(42)
random.seed(42)

In [ ]:
# Define the freight carrier selection problem data (same as previous tiers)

# Carriers and Lanes
carriers = ['Carrier 1', 'Carrier 2', 'Carrier 3']
lanes = ['Lane 1 (NYC-CHI)', 'Lane 2 (LAX-DAL)']

# Bid prices (cost per unit)
bid_prices = np.array([
    [500, 450],  # Carrier 1 bids
    [480, 470],  # Carrier 2 bids
    [520, 440]   # Carrier 3 bids
])

# Reliability scores (0-100)
reliability_scores = np.array([95, 88, 92])

# Service quality scores for each carrier-lane combination
service_scores = np.array([
    [90, 85],  # Carrier 1 service scores
    [88, 92],  # Carrier 2 service scores
    [87, 90]   # Carrier 3 service scores
])

# Demand for each lane
demand = np.array([100, 80])

# Carrier capacity limits
capacity_limits = np.array([150, 120, 200])

# Weight parameters for multi-objective optimization
alpha = 0.6  # Cost weight
beta = 0.3   # Reliability weight
gamma = 0.1  # Service weight

print("Problem Data:")
print(f"Carriers: {carriers}")
print(f"Lanes: {lanes}")
print(f"Demand: {demand}")
print(f"Weight parameters (α, β, γ): ({alpha}, {beta}, {gamma})")

In [ ]:
# Simulated Annealing implementation

def calculate_objective(assignments):
    """
    Calculate the objective function value for given assignments.
    """
    total_cost = 0
    total_reliability = 0
    total_service = 0
    assigned_lanes = 0
    
    for lane_idx, carrier_idx in enumerate(assignments):
        if carrier_idx != -1:  # Lane is assigned
            total_cost += bid_prices[carrier_idx, lane_idx] * demand[lane_idx]
            total_reliability += reliability_scores[carrier_idx]
            total_service += service_scores[carrier_idx, lane_idx]
            assigned_lanes += 1
    
    # Multi-objective function (lower is better for cost, higher is better for reliability/service)
    avg_reliability = total_reliability / assigned_lanes if assigned_lanes > 0 else 0
    avg_service = total_service / assigned_lanes if assigned_lanes > 0 else 0
    
    # Convert to single objective (weighted sum)
    # Cost component (lower is better, so we use it directly)
    cost_component = total_cost
    
    # Reliability component (higher is better, so we invert it)
    reliability_component = -beta * avg_reliability * 100  # Negative because we minimize
    
    # Service component (higher is better, so we invert it)
    service_component = -gamma * avg_service * 10  # Negative because we minimize
    
    objective = cost_component + reliability_component + service_component
    return objective

def is_feasible(assignments):
    """
    Check if assignments are feasible (capacity constraints).
    """
    carrier_utilization = np.zeros(len(carriers))
    
    for lane_idx, carrier_idx in enumerate(assignments):
        if carrier_idx != -1:
            carrier_utilization[carrier_idx] += demand[lane_idx]
    
    # Check capacity constraints
    feasible = all(carrier_utilization[i] <= capacity_limits[i] for i in range(len(carriers)))
    return feasible, carrier_utilization

def generate_neighbor(current_assignments):
    """
    Generate a neighbor solution by modifying one assignment.
    """
    neighbor = current_assignments.copy()
    
    # Select a random lane to modify
    lane_idx = random.randint(0, len(neighbor) - 1)
    
    # Select a random carrier (different from current)
    current_carrier = neighbor[lane_idx]
    available_carriers = [i for i in range(len(carriers)) if i != current_carrier]
    
    if available_carriers:
        new_carrier = random.choice(available_carriers)
        neighbor[lane_idx] = new_carrier
    
    return neighbor

def simulated_annealing(initial_assignments, initial_temp=1000, cooling_rate=0.995, max_iterations=5000):
    """
    Implement Simulated Annealing for carrier selection.
    """
    start_time = time_module.time()
    
    # Initialize
    current_assignments = initial_assignments.copy()
    current_obj = calculate_objective(current_assignments)
    best_assignments = current_assignments.copy()
    best_obj = current_obj
    
    temperature = initial_temp
    iteration = 0
    
    # Track progress
    progress_history = []
    temperature_history = []
    acceptance_history = []
    
    print(f"Starting Simulated Annealing...")
    print(f"Initial temperature: {initial_temp}")
    print(f"Cooling rate: {cooling_rate}")
    print(f"Max iterations: {max_iterations}")
    
    while iteration < max_iterations and temperature > 0.01:
        
        # Generate neighbor solution
        neighbor_assignments = generate_neighbor(current_assignments)
        
        # Check feasibility
        feasible, utilization = is_feasible(neighbor_assignments)
        if not feasible:
            # Reject infeasible solution
            acceptance_history.append(0)
            iteration += 1
            continue
        
        # Calculate objective for neighbor
        neighbor_obj = calculate_objective(neighbor_assignments)
        
        # Calculate acceptance probability
        delta = neighbor_obj - current_obj
        
        if delta < 0:  # Better solution
            accept = True
        else:  # Worse solution
            # Accept with probability based on temperature
            acceptance_prob = np.exp(-delta / temperature)
            accept = random.random() < acceptance_prob
        
        # Accept or reject
        if accept:
            current_assignments = neighbor_assignments
            current_obj = neighbor_obj
            
            # Update best solution
            if current_obj < best_obj:
                best_assignments = current_assignments.copy()
                best_obj = current_obj
        
        # Track progress
        progress_history.append(current_obj)
        temperature_history.append(temperature)
        acceptance_history.append(1 if accept else 0)
        
        # Cool down
        temperature *= cooling_rate
        iteration += 1
        
        # Progress report
        if iteration % 1000 == 0:
            print(f"Iteration {iteration}: Best Obj = {best_obj:.2f}, Temp = {temperature:.4f}")
    
    computation_time = time_module.time() - start_time
    
    return {
        'best_assignments': best_assignments,
        'best_objective': best_obj,
        'final_cost': -best_obj + (beta * 90 * 100) + (gamma * 88 * 10),  # Convert back to cost
        'iterations': iteration,
        'computation_time': computation_time,
        'progress_history': progress_history,
        'temperature_history': temperature_history,
        'acceptance_history': acceptance_history
    }

# Create initial solution (greedy approach)
def greedy_initial_solution():
    """
    Create initial solution using greedy approach.
    """
    assignments = [-1] * len(lanes)
    carrier_utilization = np.zeros(len(carriers))
    
    # Process lanes in order of demand (high to low)
    lane_order = np.argsort(-demand)
    
    for lane_idx in lane_order:
        best_carrier = -1
        best_score = -1
        
        for carrier_idx in range(len(carriers)):
            if carrier_utilization[carrier_idx] + demand[lane_idx] <= capacity_limits[carrier_idx]:
                cost = bid_prices[carrier_idx, lane_idx]
                reliability = reliability_scores[carrier_idx]
                service = service_scores[carrier_idx, lane_idx]
                
                score = (alpha * (1/cost) * 1000 + 
                         beta * reliability/100 + 
                         gamma * service/100)
                
                if score > best_score:
                    best_score = score
                    best_carrier = carrier_idx
        
        if best_carrier != -1:
            assignments[lane_idx] = best_carrier
            carrier_utilization[best_carrier] += demand[lane_idx]
    
    return assignments

# Run Simulated Annealing
initial_solution = greedy_initial_solution()
sa_result = simulated_annealing(initial_solution)

print("\nSimulated Annealing Results:")
for i, lane_idx in enumerate(np.argsort(-demand)):
    carrier_idx = sa_result['best_assignments'][lane_idx]
    if carrier_idx != -1:
        print(f"{lanes[lane_idx]}: {carriers[carrier_idx]}")
    else:
        print(f"{lanes[lane_idx]}: UNASSIGNED")

print(f"\nFinal Cost: ${sa_result['final_cost']:,.2f}")
print(f"Iterations: {sa_result['iterations']}")
print(f"Computation Time: {sa_result['computation_time']:.3f} seconds")

In [ ]:
# Comprehensive visualization of Simulated Annealing results

fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle('Simulated Annealing Optimization Results', fontsize=16, fontweight='bold')

# 1. Convergence plot
ax1 = axes[0, 0]
iterations = range(len(sa_result['progress_history']))
ax1.plot(iterations, sa_result['progress_history'], 'b-', alpha=0.7, linewidth=1)
ax1.set_xlabel('Iteration')
ax1.set_ylabel('Objective Value')
ax1.set_title('Convergence Progress')
ax1.grid(True, alpha=0.3)

# Add moving average for smoother visualization
window_size = min(100, len(sa_result['progress_history']) // 10)
if len(sa_result['progress_history']) > window_size:
    moving_avg = pd.Series(sa_result['progress_history']).rolling(window=window_size).mean()
    ax1.plot(iterations, moving_avg, 'r-', linewidth=2, label=f'Moving Avg (window={window_size})')
    ax1.legend()

# 2. Temperature decay
ax2 = axes[0, 1]
ax2.plot(iterations, sa_result['temperature_history'], 'orange', alpha=0.7, linewidth=1)
ax2.set_xlabel('Iteration')
ax2.set_ylabel('Temperature')
ax2.set_title('Temperature Schedule')
ax2.grid(True, alpha=0.3)
ax2.set_yscale('log')

# 3. Acceptance rate over time
ax3 = axes[1, 0]
# Calculate rolling acceptance rate
window = min(500, len(sa_result['acceptance_history']) // 5)
acceptance_rate = pd.Series(sa_result['acceptance_history']).rolling(window).mean()
# Ensure same length for plotting
plot_iterations = range(len(acceptance_rate))
ax3.plot(plot_iterations, acceptance_rate, 'green', alpha=0.7, linewidth=1)
ax3.set_xlabel('Iteration')
ax3.set_ylabel('Acceptance Rate')
ax3.set_title(f'Acceptance Rate (Rolling Window: {window})')
ax3.grid(True, alpha=0.3)
ax3.set_ylim(0, 1)

# 4. Solution comparison
ax4 = axes[1, 1]
# Compare with greedy and optimal solutions
methods = ['Optimal (Tier 1)', 'Greedy (Tier 2)', 'Simulated Annealing (Tier 3)']
costs = [76480, 85200, sa_result['final_cost']]  # From previous tiers
colors = ['lightgreen', 'lightblue', 'lightcoral']

bars = ax4.bar(methods, costs, color=colors, alpha=0.8)
ax4.set_ylabel('Total Cost ($)')
ax4.set_title('Solution Cost Comparison')
ax4.grid(True, alpha=0.3)

# Add cost values and improvement percentages on bars
for i, (bar, cost) in enumerate(zip(bars, costs)):
    height = bar.get_height()
    ax4.text(bar.get_x() + bar.get_width()/2., height + 1000,
             f'${cost:,.0f}', ha='center', va='bottom', fontweight='bold')

    # Calculate improvement over greedy for SA
    if i == 2:  # Simulated Annealing
        improvement = ((costs[1] - cost) / costs[1]) * 100
        ax4.text(bar.get_x() + bar.get_width()/2., height + 3000,
                f'Improvement: {improvement:.1f}%', ha='center', va='bottom',
                color='red', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Performance analysis and comparison with previous tiers

def analyze_sa_performance():
    """
    Analyze the performance of Simulated Annealing.
    """
    progress = sa_result['progress_history']
    
    print("=" * 70)
    print("SIMULATED ANNEALING - PERFORMANCE ANALYSIS")
    print("=" * 70)
    
    # Basic statistics
    print(f"\n📊 BASIC STATISTICS:")
    print(f"   Total Iterations: {sa_result['iterations']}")
    print(f"   Computation Time: {sa_result['computation_time']:.3f} seconds")
    print(f"   Final Objective: {sa_result['best_objective']:.2f}")
    print(f"   Final Cost: ${sa_result['final_cost']:,.2f}")
    print(f"   Initial Temperature: 1000.0")
    print(f"   Final Temperature: {sa_result['temperature_history'][-1]:.4f}")
    
    # Convergence analysis
    print(f"\n📈 CONVERGENCE ANALYSIS:")
    best_obj = min(progress)
    worst_obj = max(progress)
    improvement = (worst_obj - best_obj) / worst_obj * 100
    
    print(f"   Best Objective: {best_obj:.2f}")
    print(f"   Worst Objective: {worst_obj:.2f}")
    print(f"   Improvement: {improvement:.2f}%")
    
    # Find convergence point (when improvement becomes small)
    convergence_threshold = 0.01 * worst_obj  # 1% of worst objective
    convergence_point = None
    
    for i in range(len(progress) - 100):
        recent_obj = progress[i:i+100]
        if max(recent_obj) - min(recent_obj) < convergence_threshold:
            convergence_point = i + 100
            break
    
    if convergence_point:
        print(f"   Convergence Point: Iteration {convergence_point}")
        print(f"   Convergence Rate: {convergence_point / len(progress) * 100:.1f}% of total iterations")
    else:
        print(f"   Convergence Point: Not reached within {len(progress)} iterations")
    
    # Calculate acceptance rate statistics
    total_acceptances = sum(sa_result['acceptance_history'])
    acceptance_rate = total_acceptances / len(sa_result['acceptance_history'])
    
    print(f"\n🎲 ACCEPTANCE STATISTICS:")
    print(f"   Total Acceptances: {total_acceptances}/{len(sa_result['acceptance_history'])}")
    print(f"   Overall Acceptance Rate: {acceptance_rate * 100:.1f}%")
    
    # Early vs late acceptance rate
    mid_point = len(sa_result['acceptance_history']) // 2
    early_rate = sum(sa_result['acceptance_history'][:mid_point]) / mid_point
    late_rate = sum(sa_result['acceptance_history'][mid_point:]) / (len(sa_result['acceptance_history']) - mid_point)
    
    print(f"   Early Acceptance Rate (first half): {early_rate * 100:.1f}%")
    print(f"   Late Acceptance Rate (second half): {late_rate * 100:.1f}%")
    print(f"   Acceptance Decline: {(early_rate - late_rate) * 100:.1f} percentage points")
    
    # Temperature analysis
    print(f"\n🌡️ TEMPERATURE ANALYSIS:")
    initial_temp = sa_result['temperature_history'][0]
    final_temp = sa_result['temperature_history'][-1]
    temp_reduction = (initial_temp - final_temp) / initial_temp * 100
    
    print(f"   Initial Temperature: {initial_temp:.2f}")
    print(f"   Final Temperature: {final_temp:.6f}")
    print(f"   Temperature Reduction: {temp_reduction:.2f}%")
    print(f"   Cooling Rate: 0.995 per iteration")

# Run performance analysis
analyze_sa_performance()

In [ ]:
# Parameter sensitivity analysis

def test_parameter_sensitivity():
    """
    Test sensitivity to different parameter settings.
    """
    
    test_configs = [
        {'initial_temp': 500, 'cooling_rate': 0.99, 'max_iterations': 3000},
        {'initial_temp': 1000, 'cooling_rate': 0.995, 'max_iterations': 5000},  # Default
        {'initial_temp': 2000, 'cooling_rate': 0.998, 'max_iterations': 7000},
        {'initial_temp': 1500, 'cooling_rate': 0.992, 'max_iterations': 4000},
    ]
    
    results = []
    
    for i, config in enumerate(test_configs):
        print(f"\nTesting Configuration {i+1}: {config}")
        
        # Run SA with this configuration
        result = simulated_annealing(
            initial_solution, 
            initial_temp=config['initial_temp'],
            cooling_rate=config['cooling_rate'],
            max_iterations=config['max_iterations']
        )
        
        results.append({
            'config': f"T0={config['initial_temp']}, α={config['cooling_rate']}, max_iter={config['max_iterations']}",
            'final_cost': result['final_cost'],
            'computation_time': result['computation_time'],
            'iterations': result['iterations'],
            'acceptance_rate': sum(result['acceptance_history']) / len(result['acceptance_history'])
        })
        
        print(f"   Final Cost: ${result['final_cost']:,.2f}")
        print(f"   Computation Time: {result['computation_time']:.3f}s")
        print(f"   Acceptance Rate: {results[-1]['acceptance_rate'] * 100:.1f}%")
    
    return pd.DataFrame(results)

# Run sensitivity analysis
sensitivity_results = test_parameter_sensitivity()

print("\n" + "=" * 70)
print("PARAMETER SENSITIVITY ANALYSIS RESULTS")
print("=" * 70)
print(sensitivity_results.to_string(index=False))

# Visualize sensitivity results
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('Parameter Sensitivity Analysis', fontsize=16, fontweight='bold')

# Cost comparison
ax1.bar(range(len(sensitivity_results)), sensitivity_results['final_cost'], 
        color='lightcoral', alpha=0.8)
ax1.set_xlabel('Configuration')
ax1.set_ylabel('Final Cost ($)')
ax1.set_title('Cost Performance by Configuration')
ax1.set_xticks(range(len(sensitivity_results)))
ax1.set_xticklabels([f"Config {i+1}" for i in range(len(sensitivity_results))], rotation=45)
ax1.grid(True, alpha=0.3)

# Add cost values on bars
for i, cost in enumerate(sensitivity_results['final_cost']):
    ax1.text(i, cost + 500, f'${cost:,.0f}', ha='center', va='bottom', fontweight='bold')

# Computation time comparison
ax2.bar(range(len(sensitivity_results)), sensitivity_results['computation_time'], 
        color='lightgreen', alpha=0.8)
ax2.set_xlabel('Configuration')
ax2.set_ylabel('Computation Time (seconds)')
ax2.set_title('Computation Time by Configuration')
ax2.set_xticks(range(len(sensitivity_results)))
ax2.set_xticklabels([f"Config {i+1}" for i in range(len(sensitivity_results))], rotation=45)
ax2.grid(True, alpha=0.3)

# Add time values on bars
for i, time in enumerate(sensitivity_results['computation_time']):
    ax2.text(i, time + 0.05, f'{time:.3f}s', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

### Why this Tier exists vs previous approaches
Simulated Annealing provides advantages over both mathematical optimization and greedy heuristics:
- **Global Search**: Can escape local optima unlike greedy methods
- **Adaptive Exploration**: Temperature controls exploration vs exploitation balance
- **Probabilistic Acceptance**: Allows temporary moves to worse solutions
- **Convergence Control**: Systematic approach to finding high-quality solutions

### Pros / Cons vs previous tiers
**Pros vs Mathematical Optimization:**
- Faster computation time for large problems
- No requirement for specialized optimization solvers
- More flexible problem formulation
- Better scalability to complex constraints

**Pros vs Greedy Heuristic:**
- Can escape local optima
- Systematic exploration of solution space
- Better solution quality typically
- Adjustable search intensity

**Cons:**
- No guarantee of global optimality
- Parameter tuning required
- Computationally more expensive than greedy
- Random nature means results can vary

### When to use this Tier
- **Medium-sized problems** where exact optimization is too slow
- **Complex constraints** that make mathematical formulation difficult
- **When solution quality matters more than speed** but exact optimization is impractical
- **Multi-modal optimization landscapes** with many local optima
- **When you need a balance** between solution quality and computation time

In [ ]:
# Final summary and Simulated Annealing assessment
print("=" * 70)
print("SIMULATED ANNEALING - FINAL ASSESSMENT")
print("=" * 70)

print(f"\n🎯 OPTIMIZATION RESULTS:")
for i, lane_idx in enumerate(np.argsort(-demand)):
    carrier_idx = sa_result['best_assignments'][lane_idx]
    if carrier_idx != -1:
        print(f"   {lanes[lane_idx]}: {carriers[carrier_idx]}")
    else:
        print(f"   {lanes[lane_idx]}: UNASSIGNED")

print(f"\n📊 PERFORMANCE METRICS:")
print(f"   Final Cost: ${sa_result['final_cost']:,.2f}")
print(f"   Iterations Completed: {sa_result['iterations']}")
print(f"   Computation Time: {sa_result['computation_time']:.3f} seconds")
print(f"   Best Objective: {sa_result['best_objective']:.2f}")

print(f"\n🌡️ ALGORITHM CHARACTERISTICS:")
print(f"   ✓ Temperature-controlled search process")
print(f"   ✓ Probabilistic acceptance of worse solutions")
print(f"   ✓ Systematic exploration and exploitation balance")
print(f"   ✓ Convergence monitoring and tracking")

# Calculate performance comparison
greedy_cost = 85200  # From Tier 2
optimal_cost = 76480  # From Tier 1
improvement_over_greedy = ((greedy_cost - sa_result['final_cost']) / greedy_cost) * 100
gap_to_optimal = ((sa_result['final_cost'] - optimal_cost) / optimal_cost) * 100

print(f"\n📈 PERFORMANCE COMPARISON:")
print(f"   Improvement over Greedy: {improvement_over_greedy:.2f}%")
print(f"   Gap to Optimal: {gap_to_optimal:.2f}%")
print(f"   Quality vs Greedy: {'Better' if improvement_over_greedy > 0 else 'Worse'}")
print(f"   Quality vs Optimal: {gap_to_optimal:.1f}% from optimal")

# Acceptance statistics
acceptance_rate = sum(sa_result['acceptance_history']) / len(sa_result['acceptance_history'])
print(f"\n🎲 SEARCH STATISTICS:")
print(f"   Acceptance Rate: {acceptance_rate * 100:.1f}%")
print(f"   Temperature Range: 1000.0 → {sa_result['temperature_history'][-1]:.6f}")
print(f"   Search Intensity: {'High' if acceptance_rate > 0.5 else 'Medium' if acceptance_rate > 0.3 else 'Low'}")

print(f"\n⚠️ ALGORITHM INSIGHTS:")
print(f"   • Temperature controls exploration vs exploitation")
print(f"   • Early high temperature allows diverse exploration")
print(f"   • Late low temperature focuses on local refinement")
print(f"   • Probabilistic acceptance prevents premature convergence")
    
print(f"\n⚠️ LIMITATIONS:")
print(f"   • No guarantee of finding global optimum")
print(f"   • Parameter sensitivity (temperature, cooling rate)")
print(f"   • Random nature affects reproducibility")
print(f"   • Computation time grows with problem complexity")
    
print(f"\n🔮 PRACTICAL APPLICATIONS:")
print(f"   ✓ Complex combinatorial optimization problems")
print(f"   ✓ Multi-modal landscapes with many local optima")
print(f"   ✓ Medium-sized problems where exact methods are slow")
print(f"   ✓ Problems with complex constraints")
    
print(f"\n💡 WHEN TO USE:")
print(f"   • Solution quality is important but exact optimization is impractical")
print(f"   • Problem has many local optima that greedy methods get stuck in")
print(f"   • You have time for parameter tuning and experimentation")
print(f"   • Moderate computation time is acceptable")
    
print("\n" + "=" * 70)
print("SIMULATED ANNEALING COMPLETE")
print("=" * 70)